# Causal Inference Agentic Workflow

A **LangGraph-based multi-agent system** that answers causal questions using
Pearl's causal ladder (Association → Intervention → Counterfactual).

### Architecture

```
User Question
     │
     ▼
┌─────────────┐
│ Orchestrator │  ← classifies rung, extracts CausalQuery
└─────┬───────┘
      │  L1 / L2 / L3
      ▼
┌──────────────────────────┐
│ L1 Association Agent     │  P(Y|X)
│ L2 Intervention Agent    │  P(Y|do(X))
│ L3 Counterfactual Agent  │  P(Y_x | X=x', Y=y')
└──────────┬───────────────┘
           ▼
┌───────────────┐
│   Validator   │  ← identifiability, positivity, estimator check
└───────┬───────┘
   pass │ re_route → Orchestrator (loop)
   fail │
        ▼
┌───────────────────┐
│ Sensitivity Agent │  ← (optional) E-values, Rosenbaum bounds
└───────┬───────────┘
        ▼
┌───────────────┐
│  Synthesizer  │  ← final causal report
└───────────────┘
```

### Agents

| Agent | Role |
|-------|------|
| **Orchestrator** | Classifies the question into L1/L2/L3, extracts treatment/outcome/estimand |
| **L1 Association** | Handles $P(Y \mid X)$ — observational analysis with confounding warnings |
| **L2 Intervention** | Handles $P(Y \mid do(X))$ — checks identifiability, selects estimator |
| **L3 Counterfactual** | Handles $P(Y_x \mid X=x', Y=y')$ — abduction-action-prediction |
| **Validator** | Checks identifiability, positivity, estimator-estimand match |
| **Sensitivity** | *(New)* Evaluates robustness via E-values, Rosenbaum bounds |
| **Synthesizer** | Produces the final structured causal report |

In [ ]:
# Optional — uncomment to install dependencies
# %pip install langgraph openai anthropic python-dotenv networkx matplotlib

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

src_dir = Path.cwd().parent
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from causal_inference.settings import Settings
from causal_inference.llm_client import build_llm_client
from causal_inference.graph import run_causal_workflow

print("Package imported from:", src_dir)

## Configuration

API keys can come from a `.env` file or be set directly below.

```bash
# .env (place in src/ or repo root)
LLM_PROVIDER=openai          # or "anthropic"
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_MODEL=gpt-4o-mini
ANTHROPIC_MODEL=claude-3-5-sonnet-latest
```

In [ ]:
# Override here if you prefer not to use .env
LLM_PROVIDER = "openai"                         # "openai" or "anthropic"
OPENAI_API_KEY_INLINE = "<your_OPENAI_API_KEY>"  # or None to use .env
ANTHROPIC_API_KEY_INLINE = None

settings = Settings.from_env()
if LLM_PROVIDER:
    settings = Settings(
        llm_provider=LLM_PROVIDER,
        openai_api_key=OPENAI_API_KEY_INLINE or settings.openai_api_key,
        anthropic_api_key=ANTHROPIC_API_KEY_INLINE or settings.anthropic_api_key,
        openai_model=settings.openai_model,
        anthropic_model=settings.anthropic_model,
    )

llm = build_llm_client(settings)
print(f"LLM provider: {settings.llm_provider} / model: {settings.openai_model if settings.llm_provider == 'openai' else settings.anthropic_model}")

---

## DAG & SCM Definitions

The agents reason over a shared **Causal DAG** and optionally a **Structural
Causal Model** (SCM) for Rung 3 analysis.

In [ ]:
# DAG for L1 / L2 examples: Smoking and Cancer with genetics as confounder
smoking_dag = {
    "nodes": ["Smoking", "Tar", "Cancer", "Genetics"],
    "edges": [
        ["Smoking", "Tar"],
        ["Tar", "Cancer"],
        ["Genetics", "Smoking"],
        ["Genetics", "Cancer"],
    ],
    "description": (
        "Smoking → Tar → Cancer (front-door path). "
        "Genetics is an unmeasured common cause of Smoking and Cancer."
    ),
}

# SCM for L3 example: drug treatment
drug_scm = {
    "structural_equations": {
        "X": "U_X",
        "Y": "0.6 * X + U_Y",
    },
    "exogenous": {
        "U_X": "Bernoulli(0.5)",
        "U_Y": "Normal(0, 0.1)",
    },
    "description": (
        "X = took drug (binary), Y = recovery score. "
        "Structural equation: Y = 0.6*X + U_Y."
    ),
}

drug_dag = {
    "nodes": ["X", "Y"],
    "edges": [["X", "Y"]],
    "description": "Simple RCT-like DAG: X → Y with no confounders.",
}

# IV scenario DAG
iv_dag = {
    "nodes": ["Education Policy (Z)", "Years of Education (X)", "Earnings (Y)", "Ability (U)"],
    "edges": [
        ["Education Policy (Z)", "Years of Education (X)"],
        ["Years of Education (X)", "Earnings (Y)"],
        ["Ability (U)", "Years of Education (X)"],
        ["Ability (U)", "Earnings (Y)"],
    ],
    "description": (
        "IV scenario: Education Policy is an instrument for the effect of "
        "Education on Earnings. Ability is an unmeasured confounder."
    ),
}

print("DAGs and SCMs defined.")

---

## Example 1 — L1 Association

> "Is smoking correlated with cancer after controlling for genetics?"

This is an **observational / associational** question → expected to route to the **L1 agent**.

In [ ]:
from IPython.display import Markdown

result_l1 = await run_causal_workflow(
    question="Is smoking correlated with cancer after controlling for genetics?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l1["ladder_rung"])
Markdown(result_l1["final_report"])

---

## Example 2 — L2 Intervention

> "What is the causal effect of smoking on cancer?"

This is an **interventional** question — $P(\text{Cancer} \mid do(\text{Smoking}))$ →
expected to route to the **L2 agent**.

The DAG has an unmeasured confounder (Genetics) but a front-door path via Tar.

In [ ]:
result_l2 = await run_causal_workflow(
    question="What is the causal effect of smoking on cancer?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l2["ladder_rung"])
Markdown(result_l2["final_report"])

---

## Example 3 — L2 with Sensitivity Analysis

> "Does education cause higher earnings?"

Running with `run_sensitivity=True` adds E-value and Rosenbaum bounds
analysis to assess robustness to unmeasured confounding.

In [ ]:
result_l2_sens = await run_causal_workflow(
    question="Does education cause higher earnings, given that ability is a potential unmeasured confounder?",
    llm=llm,
    dag=iv_dag,
    run_sensitivity=True,
)

print("Ladder rung:", result_l2_sens["ladder_rung"])
Markdown(result_l2_sens["final_report"])

---

## Example 4 — L3 Counterfactual (with SCM)

> "A patient took the drug (X=1) and recovered (Y=0.7). What would their
> recovery have been had they NOT taken the drug?"

This is a **counterfactual** question → routed to **L3**. The SCM is fully specified,
so the agent can execute the abduction–action–prediction procedure.

In [ ]:
result_l3 = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=drug_scm,
)

print("Ladder rung:", result_l3["ladder_rung"])
Markdown(result_l3["final_report"])

---

## Example 5 — L3 Counterfactual (without SCM — graceful degradation)

Same counterfactual question, but **no SCM** is provided.
The L3 agent should degrade to bounds (Manski) and communicate the limitation.

In [ ]:
result_l3_no_scm = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=None,
)

print("Ladder rung:", result_l3_no_scm["ladder_rung"])
Markdown(result_l3_no_scm["final_report"])

---

## Inspect Intermediate State

Each run returns the full `CausalState`. You can inspect every intermediate
artifact: the orchestrator's classification, the rung agent's analysis,
the validator's checks, sensitivity analysis, and the final report.

In [ ]:
import json

def inspect_state(result, title=""):
    """Pretty-print key fields from a CausalState result."""
    if title:
        print(f"=== {title} ===\n")
    for key in ("ladder_rung", "causal_query", "analysis_result",
                "validation_result", "sensitivity_result", "iteration"):
        val = result.get(key)
        if val is None or val == {}:
            continue
        print(f"── {key} ──")
        if isinstance(val, dict):
            print(json.dumps(val, indent=2))
        else:
            print(val)
        print()

inspect_state(result_l2, title="L2 Intervention Example — Full State")

In [ ]:
inspect_state(result_l3, title="L3 Counterfactual (with SCM) — Full State")

In [ ]:
# If sensitivity analysis was run, inspect it
if result_l2_sens.get("sensitivity_result"):
    inspect_state(result_l2_sens, title="L2 with Sensitivity Analysis — Full State")

---

## Visualize the DAGs

Use the built-in DAG visualization to inspect the causal structures.

In [ ]:
import matplotlib.pyplot as plt
from causal_inference.viz.dag import draw_dag_from_spec

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

draw_dag_from_spec(smoking_dag, ax=axes[0], title="Smoking DAG\n(Front-door + Confounder)",
                   layout="spring")
draw_dag_from_spec(drug_dag, ax=axes[1], title="Drug RCT DAG\n(No confounders)",
                   layout="spring")
draw_dag_from_spec(iv_dag, ax=axes[2], title="IV DAG\n(Instrument + Confounder)",
                   layout="spring")

fig.suptitle("Causal DAGs Used in This Demo", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()